# Приложение 10.2: Использование инструмента

- [Урок](#lesson)
- [Упражнения](#exercises)
- [Пример игровой площадки](#example-playground)

## Настройка

Запустите следующую ячейку настройки, чтобы загрузить ключ API и установить вспомогательную функцию get_completion.

In [ ]:
# Rewrittten to call Claude 3 Sonnet, which is generally better at tool use, and include stop_sequences
# Import python's built-in regular expression library
import re
import boto3
import json

# Import the hints module from the utils package
import os
import sys
module_path = ".."
sys.path.append(os.path.abspath(module_path))
from utils import hints

# Override the MODEL_NAME variable in the IPython store to use Sonnet instead of the Haiku model
MODEL_NAME='anthropic.claude-3-sonnet-20240229-v1:0'
%store -r AWS_REGION

client = boto3.client('bedrock-runtime',region_name=AWS_REGION)

def get_completion(messages, system_prompt="", prefill="", stop_sequences=None):
    body = json.dumps(
        {
            "anthropic_version": '',
            "max_tokens": 2000,
            "temperature": 0.0,
            "top_p": 1,
            "messages":messages,
            "system": system_prompt,
            "stop_sequences": stop_sequences
        }
    )
    response = client.invoke_model(body=body, modelId=MODEL_NAME)
    response_body = json.loads(response.get('body').read())

    return response_body.get('content')[0].get('text')

---

## Урок

Хотя на первый взгляд это может показаться концептуально сложным, использование инструментов, то есть вызов функций, на самом деле довольно просто! Вы уже владеете всеми навыками, необходимыми для реализации использования инструментов, что на самом деле представляет собой просто комбинацию замены и цепочки подсказок.

В предыдущих упражнениях по замене мы заменяли текст в подсказках. При использовании инструмента мы заменяем результаты инструмента или функции в подсказки. Клод не может буквально вызывать инструменты и функции или получать к ним доступ. Вместо этого у нас есть Клод:
1. Выведите имя инструмента и аргументы, которые он хочет вызвать.
2. Остановите дальнейшее формирование ответов во время вызова инструмента.
3. Затем мы повторно выводим запрос с добавленными результатами инструмента.

Вызов функций полезен, поскольку расширяет возможности Claude и позволяет Claude выполнять гораздо более сложные многоэтапные задачи.
Несколько примеров функций, которые можно дать Клоду:
- Калькулятор
- Счетчик слов
- SQL запросы к базе данных и извлечение данных
- API погоды

Вы можете заставить Клода использовать инструменты, объединив эти два элемента:

1. Системное приглашение, в котором мы даем Клоду объяснение концепции использования инструмента, а также подробный описательный список инструментов, к которым он имеет доступ.
2. Логика управления, с помощью которой можно организовать и выполнить запросы на использование инструмента Клода.

### План использования инструмента

*В этом уроке рассматривается наш текущий формат использования инструментов. Однако в ближайшем будущем мы будем обновлять и улучшать функциональные возможности использования инструментов, в том числе:*
* *Более упрощенный формат определений и вызовов функций*
* *Более надежная обработка ошибок и покрытие крайних случаев*
* *Более тесная интеграция с остальной частью нашего API*
* *Повышенная надежность и производительность, особенно для более сложных задач использования инструмента*

### Примеры

Чтобы включить использование инструментов в Claude, мы начинаем с системной подсказки. В этом специальном инструменте используйте системную подсказку и сообщите Клоду:
* Основная предпосылка использования инструмента и что это влечет за собой.
* Как Клод может вызывать и использовать предоставленные ему инструменты
* Подробный список инструментов, к которым у него есть доступ в этом конкретном сценарии. 

Вот первая часть системного приглашения, объясняющая Клоду использование инструмента. Эту часть системного приглашения можно обобщить для всех случаев, когда Клоду предлагается использовать инструмент. Структура вызова инструмента, которую мы даем Клоду (`<function_calls> [...] </function_calls>`), представляет собой структуру, которую Клод специально обучен использовать, поэтому мы рекомендуем вам придерживаться ее.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

Вот вторая часть системного приглашения, в которой определяются точные инструменты, к которым Клод имеет доступ в этой конкретной ситуации. В этом примере мы предоставим Клоду инструмент-калькулятор, который принимает три параметра: два операнда и оператор. 

Затем объединяем две части системной подсказки.

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

Теперь мы можем задать Клоду вопрос, требующий использования инструмента calculator. Мы будем использовать <function_calls\> в stop_sequences, чтобы определить, вызывает ли Клод функцию и когда это происходит.

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Теперь мы можем извлечь параметры из вызова функции Клода и фактически запустить функцию от имени Клода.

Сначала мы определим код функции.

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

Затем мы извлечем параметры из ответа на вызов функции Клода. Если все параметры существуют, запускаем калькулятор.

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

Теперь, когда у нас есть результат, нам нужно его правильно отформатировать, чтобы, когда мы передадим его обратно Клоду, Клод понял, к какому инструменту относится этот результат. Для этого существует установленный формат, который Клод научился распознавать:```
<function_results>
<result>
<tool_name>{TOOL_NAME}</tool_name>
<stdout>
{TOOL_RESULT}
</stdout>
</result>
</function_results>
```Запустите ячейку ниже, чтобы отформатировать приведенный выше результат инструмента в эту структуру.

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

Теперь все, что нам нужно сделать, это отправить этот результат обратно Клоду, добавив его в ту же цепочку сообщений, что и раньше, и все готово!

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construct the full conversation
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Print Claude's response
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

Поздравляем с запуском всей цепочки использования инструмента!

А что, если мы зададим Клоду вопрос, который вообще не требует использования данного инструмента?

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

Успех! Как видите, Клод знал, что не следует вызывать функцию, когда в ней нет необходимости.

Если вы хотите поэкспериментировать с подсказками к уроку, не меняя никакого содержания выше, прокрутите блокнот урока до конца, чтобы перейти к [**Пример игровой площадки**](#example-playground).

---

## Упражнения
- [Упражнение 10.2.1. SQL](#exercise-1021---SQL)

### Упражнение 10.2.1 — SQL
В этом упражнении вы напишете подсказку об использовании инструмента для выполнения запросов и записи в самую маленькую в мире «базу данных». Вот инициализированная база данных, которая на самом деле представляет собой просто словарь.

In [ ]:
db = {
    "users": [
        {"id": 1, "name": "Alice", "email": "alice@example.com"},
        {"id": 2, "name": "Bob", "email": "bob@example.com"},
        {"id": 3, "name": "Charlie", "email": "charlie@example.com"}
    ],
    "products": [
        {"id": 1, "name": "Widget", "price": 9.99},
        {"id": 2, "name": "Gadget", "price": 14.99},
        {"id": 3, "name": "Doohickey", "price": 19.99}
    ]
}

А вот код функций, которые записывают данные в базу данных и из нее.

In [ ]:
def get_user(user_id):
    for user in db["users"]:
        if user["id"] == user_id:
            return user
    return None

def get_product(product_id):
    for product in db["products"]:
        if product["id"] == product_id:
            return product
    return None

def add_user(name, email):
    user_id = len(db["users"]) + 1
    user = {"id": user_id, "name": name, "email": email}
    db["users"].append(user)
    return user

def add_product(name, price):
    product_id = len(db["products"]) + 1
    product = {"id": product_id, "name": name, "price": price}
    db["products"].append(product)
    return product

Чтобы выполнить упражнение, начните с определения системного приглашения, например «system_prompt_tools_specific_tools», приведенного выше. Обязательно укажите имя и описание каждого инструмента, а также имя, тип и описание каждого параметра для каждой функции. Ниже мы предоставили вам некоторые стартовые леса.

In [ ]:
system_prompt_tools_specific_tools_sql = """
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools_sql

Когда вы будете готовы, вы можете опробовать подсказку системы определения инструмента на примерах ниже. Просто запустите ячейку ниже!

In [ ]:
examples = [
    "Add a user to the database named Deborah.",
    "Add a product to the database named Thingo",
    "Tell me the name of User 2",
    "Tell me the name of Product 3"
]

for example in examples:
    message = {
        "role": "user",
        "content": example
    }

    # Get & print Claude's response
    function_calling_response = get_completion([message], system_prompt=system_prompt, stop_sequences=stop_sequences)
    print(example, "\n----------\n\n", function_calling_response, "\n*********\n*********\n*********\n\n")

Если вы все сделали правильно, сообщения о вызове функции должны правильно вызывать функции add_user, add_product, get_user и get_product.

Чтобы получить дополнительную оценку, добавьте несколько ячеек кода и напишите код анализа параметров. Затем вызовите функции с параметрами, которые дает вам Клод, чтобы увидеть состояние «базы данных» после вызова.

❓Если хотите увидеть возможное решение, запустите ячейку ниже!

In [ ]:
print(hints.exercise_10_2_1_solution)

### Поздравляю!

Поздравляем с изучением использования инструментов и вызова функций! Если вы хотите узнать больше о поиске и RAG, перейдите к последнему разделу приложения.

---

## Пример игровой площадки

Это область, где вы можете свободно экспериментировать с примерами подсказок, представленными в этом уроке, и настраивать подсказки, чтобы увидеть, как они могут повлиять на ответы Клода.

In [ ]:
system_prompt_tools_general_explanation = """You have access to a set of functions you can use to answer the user's question. This includes access to a
sandboxed computing environment. You do NOT currently have the ability to inspect files or interact with external
resources, except by invoking the below functions.

You can invoke one or more functions by writing a "<function_calls>" block like the following as part of your
reply to the user:
<function_calls>
<invoke name="$FUNCTION_NAME">
<antml:parameter name="$PARAMETER_NAME">$PARAMETER_VALUE</parameter>
...
</invoke>
<nvoke name="$FUNCTION_NAME2">
...
</invoke>
</function_calls>

String and scalar parameters should be specified as is, while lists and objects should use JSON format. Note that
spaces for string values are not stripped. The output is not expected to be valid XML and is parsed with regular
expressions.

The output and/or any errors will appear in a subsequent "<function_results>" block, and remain there as part of
your reply to the user.
You may then continue composing the rest of your reply to the user, respond to any errors, or make further function
calls as appropriate.
If a "<function_results>" does NOT appear after your function calls, then they are likely malformatted and not
recognized as a call."""

In [ ]:
system_prompt_tools_specific_tools = """Here are the functions available in JSONSchema format:
<tools>
<tool_description>
<tool_name>calculator</tool_name>
<description>
Calculator function for doing basic arithmetic.
Supports addition, subtraction, multiplication
</description>
<parameters>
<parameter>
<name>first_operand</name>
<type>int</type>
<description>First operand (before the operator)</description>
</parameter>
<parameter>
<name>second_operand</name>
<type>int</type>
<description>Second operand (after the operator)</description>
</parameter>
<parameter>
<name>operator</name>
<type>str</type>
<description>The operation to perform. Must be either +, -, *, or /</description>
</parameter>
</parameters>
</tool_description>
</tools>
"""

system_prompt = system_prompt_tools_general_explanation + system_prompt_tools_specific_tools

In [ ]:
multiplication_message = {
    "role": "user",
    "content": "Multiply 1,984,135 by 9,343,116"
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)

In [ ]:
def do_pairwise_arithmetic(num1, num2, operation):
    if operation == '+':
        return num1 + num2
    elif operation == "-":
        return num1 - num2
    elif operation == "*":
        return num1 * num2
    elif operation == "/":
        return num1 / num2
    else:
        return "Error: Operation not supported."

In [ ]:
def find_parameter(message, parameter_name):
    parameter_start_string = f"name=\"{parameter_name}\">"
    start = message.index(parameter_start_string)
    if start == -1:
        return None
    if start > 0:
        start = start + len(parameter_start_string)
        end = start
        while message[end] != "<":
            end += 1
    return message[start:end]

first_operand = find_parameter(function_calling_response, "first_operand")
second_operand = find_parameter(function_calling_response, "second_operand")
operator = find_parameter(function_calling_response, "operator")

if first_operand and second_operand and operator:
    result = do_pairwise_arithmetic(int(first_operand), int(second_operand), operator)
    print("---------------- RESULT ----------------")
    print(f"{result:,}")

In [ ]:
def construct_successful_function_run_injection_prompt(invoke_results):
    constructed_prompt = (
        "<function_results>\n"
        + '\n'.join(
            f"<result>\n<tool_name>{res['tool_name']}</tool_name>\n<stdout>\n{res['tool_result']}\n</stdout>\n</result>"
            for res in invoke_results
        ) + "\n</function_results>"
    )

    return constructed_prompt

formatted_results = [{
    'tool_name': 'do_pairwise_arithmetic',
    'tool_result': result
}]
function_results = construct_successful_function_run_injection_prompt(formatted_results)
print(function_results)

In [ ]:
full_first_response = function_calling_response + "</function_calls>"

# Construct the full conversation
messages = [multiplication_message,
{
    "role": "assistant",
    "content": full_first_response
},
{
    "role": "user",
    "content": function_results
}]
   
# Print Claude's response
final_response = get_completion(messages, system_prompt=system_prompt, stop_sequences=stop_sequences)
print("------------- FINAL RESULT -------------")
print(final_response)

In [ ]:
non_multiplication_message = {
    "role": "user",
    "content": "Tell me the capital of France."
}

stop_sequences = ["</function_calls>"]

# Get Claude's response
function_calling_response = get_completion([non_multiplication_message], system_prompt=system_prompt, stop_sequences=stop_sequences)
print(function_calling_response)